<div style="border:2px solid #1f77b4;border-radius:10px;padding:12px;background:#f4f8ff">
<b>Version:</b> <code>v1.1-fiscal-period-alignment</code><br>
<b>Updated:</b> <code>2026-09-16 22:30 CEST (UTC+02:00)</code><br>
<b>Branch:</b> <code>main</code><br>
<b>Fix:</b> fiscal period-end comparison + mandatory structured financial tasks.
</div>

# Chapter 5 - Lab 2: <font color='blue'>The Full Deep Search Pipeline</font>

**Plan → Execute → Validate → (Replan?) → Synthesize**. This Colab lab uses PydanticAI, `asyncio` and stub tools for NVDA, AMD and INTC.

## 1. Install packages and load shared models

In [ ]:
%pip install -q pydantic-ai pydantic python-dotenv

In [ ]:
import os, urllib.request
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    pass

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%205/common.py',
        'common.py',
    )

from common import (
    ResearchPlan, SubTask, TaskStatus, CompanyMetrics, ValidationResult,
    ResearchReport, PLANNING_MODEL, SYNTHESIS_MODEL, MAX_REPLAN_ATTEMPTS,
    extract_ticker, format_plan, format_validation,
)
from pydantic_ai import Agent
print('Shared models loaded.')

## 2. Stub tools

`period` now stores the real fiscal **period-end date**. Fiscal-year labels are company-specific, so the validator compares dates and allows a 45-day window.

In [ ]:
STUB_METRICS = {
    'NVDA': CompanyMetrics(ticker='NVDA', company_name='NVIDIA',
        revenue=60_922, revenue_growth=125.8, eps=12.96, pe_ratio=72.4,
        gross_margin=72.7, operating_margin=54.1, market_cap=2891.0,
        period='2024-01-28'),
    'AMD': CompanyMetrics(ticker='AMD', company_name='Advanced Micro Devices',
        revenue=22_680, revenue_growth=-4.4, eps=0.53, pe_ratio=233.0,
        gross_margin=46.1, operating_margin=1.8, market_cap=190.0,
        period='2023-12-30'),
    'INTC': CompanyMetrics(ticker='INTC', company_name='Intel',
        revenue=54_228, revenue_growth=-13.7, eps=0.40, pe_ratio=109.0,
        gross_margin=40.0, operating_margin=-1.8, market_cap=130.0,
        period='2023-12-30'),
}

async def get_financial_metrics(ticker):
    if ticker not in STUB_METRICS:
        raise ValueError(f'No stub data for {ticker}')
    return STUB_METRICS[ticker]

async def get_risk_factors(ticker):
    return f'[Risk factors for {ticker} — stub]\n- AI demand cyclicality\n- Geopolitical risk\n- Manufacturing concentration'

async def search_financial_news(query):
    return f'[News search for: {query!r} — stub]\nRecent reports indicate continued AI infrastructure demand.'

## 3. Planner

For every company whose financial performance is compared, the planner must create a no-dependency `financial_api` task. SEC/web evidence can complement, but not replace, structured `CompanyMetrics`.

In [ ]:
PLANNER_SYSTEM_PROMPT = (
    'You are a financial research planner. Decompose the question into 5-10 '
    'concrete sub-tasks with explicit dependencies. Sources: financial_api, '
    'sec_filing, web_search. For every company whose financial performance is '
    'compared, create a no-dependency financial_api task returning structured '
    'financial metrics. Do not substitute sec_filing or web_search for those '
    'metrics. Use SEC for filing/risk evidence and web_search for recent context. '
    'Independent tasks should run in parallel; comparative tasks depend on the '
    'relevant data tasks; final synthesis depends on the required analyses.'
)
planner_agent = Agent(model=PLANNING_MODEL, system_prompt=PLANNER_SYSTEM_PROMPT,
                      output_type=ResearchPlan, retries=2)

## 4. Researcher — dependency-aware parallel execution

In [ ]:
import asyncio

analyst_agent = Agent(
    model=SYNTHESIS_MODEL,
    system_prompt='You are a senior financial analyst. Use provided evidence and specific numbers. Never fabricate missing data.',
    retries=1,
)

async def _route_and_execute(task, prior):
    s = task.data_sources
    if 'financial_api' in s and not task.dependencies:
        ticker = extract_ticker(task.description)
        if ticker:
            return (await get_financial_metrics(ticker)).model_dump_json(indent=2)
    if 'sec_filing' in s and 'financial_api' not in s:
        ticker = extract_ticker(task.description)
        if ticker:
            return await get_risk_factors(ticker)
    if 'web_search' in s and len(s) == 1:
        return await search_financial_news(task.description)

    context = '\n\n---\n\n'.join(prior[d] for d in task.dependencies if d in prior)
    return (await analyst_agent.run(
        f'Task: {task.description}\n\nAvailable data:\n{context or "No prior data."}'
    )).output

async def execute_plan(plan):
    results = {}
    by_id = {t.id: t for t in plan.sub_tasks}
    while any(t.status == TaskStatus.PENDING for t in plan.sub_tasks):
        ready = [t for t in plan.sub_tasks
                 if t.status == TaskStatus.PENDING
                 and all(by_id[d].status == TaskStatus.COMPLETED for d in t.dependencies)]
        if not ready:
            for t in [x for x in plan.sub_tasks if x.status == TaskStatus.PENDING]:
                t.status, t.result = TaskStatus.FAILED, 'Unresolvable dependencies'
                results[t.id] = t.result
            break

        async def run_one(t):
            t.status = TaskStatus.IN_PROGRESS
            try:
                r = await _route_and_execute(t, results)
                t.status, t.result = TaskStatus.COMPLETED, r
            except Exception as e:
                t.status, t.result = TaskStatus.FAILED, f'[FAILED] {e}'
            return t.id, t.result

        for task_id, result in await asyncio.gather(*(run_one(t) for t in ready)):
            results[task_id] = result
            print(f'  [{by_id[task_id].status.value}] task {task_id}')
    return results

## 5. Validator — deterministic checks

In [ ]:
import math
from datetime import date

def _validate_metrics(m, errors, warnings):
    if not m.period:
        errors.append(f'{m.ticker}: missing reported fiscal period')
    fields = ('revenue','revenue_growth','eps','gross_margin','operating_margin','market_cap')
    for f in fields:
        v = getattr(m, f)
        if v is None or not math.isfinite(v):
            errors.append(f'{m.ticker}: {f} must be finite')
    if m.pe_ratio is None:
        if m.eps > 0:
            errors.append(f'{m.ticker}: pe_ratio must be finite')
    elif not math.isfinite(m.pe_ratio):
        errors.append(f'{m.ticker}: pe_ratio must be finite')
    elif m.pe_ratio < 0 or m.pe_ratio > 300:
        warnings.append(f'{m.ticker}: unusual P/E ratio ({m.pe_ratio})')
    if m.gross_margin is not None and not (-10 <= m.gross_margin <= 100):
        errors.append(f'{m.ticker}: gross margin outside plausible range')
    if m.operating_margin is not None and not (-100 <= m.operating_margin <= 100):
        errors.append(f'{m.ticker}: operating margin outside plausible range')
    if (m.gross_margin is not None and m.operating_margin is not None
            and m.gross_margin > 5 and m.operating_margin > m.gross_margin + 1):
        errors.append(f'{m.ticker}: operating margin exceeds gross margin')

def _period_date(period):
    try:
        return date.fromisoformat(period) if period else None
    except ValueError:
        return None

def _validate_cross_company(metrics, warnings):
    dated = [(m.ticker, _period_date(m.period)) for m in metrics]
    if all(d is not None for _, d in dated):
        dates = [d for _, d in dated]
        spread = (max(dates) - min(dates)).days
        if spread > 45:
            labels = ', '.join(f'{t}={d.isoformat()}' for t, d in dated)
            warnings.append(
                f'Fiscal period end dates differ by {spread} days: {labels}. '
                'Comparisons may not be apples-to-apples.'
            )
        return
    periods = {m.period for m in metrics}
    if len(periods) > 1:
        labels = ', '.join(f'{m.ticker}={m.period}' for m in metrics)
        warnings.append(f'Fiscal period mismatch across companies: {labels}.')

def validate_research(plan, results, required_tickers):
    errors, warnings, gaps = [], [], []
    incomplete = [t for t in plan.sub_tasks if t.status != TaskStatus.COMPLETED]
    for t in incomplete:
        gaps.append(f'Task {t.id} is not completed: {t.description}')

    parsed, found = [], set()
    for t in plan.sub_tasks:
        if t.status != TaskStatus.COMPLETED:
            continue
        r = results.get(t.id)
        if not isinstance(r, str) or not r.strip():
            gaps.append(f'Task {t.id} has no result')
            continue
        try:
            m = CompanyMetrics.model_validate_json(r)
        except (ValueError, TypeError):
            if 'financial_api' in t.data_sources and not t.dependencies:
                errors.append(f'Task {t.id} did not return valid financial metrics')
            continue
        parsed.append(m)
        found.add(m.ticker.upper())
        _validate_metrics(m, errors, warnings)

    for ticker in sorted(set(x.upper() for x in required_tickers) - found):
        gaps.append(f'Missing structured financial data for required ticker: {ticker}')
    if len(parsed) >= 2:
        _validate_cross_company(parsed, warnings)

    return ValidationResult(
        is_valid=not errors and not gaps,
        errors=errors, warnings=warnings, gaps=gaps,
    )

## 6. Synthesizer

In [ ]:
synthesizer_agent = Agent(
    model=SYNTHESIS_MODEL,
    system_prompt=(
        'Produce a structured equity-research memo using only supplied evidence. '
        'Use specific numbers, acknowledge warnings, never fabricate missing data.'
    ),
    output_type=ResearchReport,
    retries=2,
)

async def synthesize_report(question, results, validation):
    findings = '\n\n'.join(
        f'=== Task {i} Result ===\n{r}' for i, r in sorted(results.items())
    )
    if validation.warnings:
        findings += '\n\n=== Warnings ===\n' + '\n'.join(
            f'- {w}' for w in validation.warnings
        )
    return (await synthesizer_agent.run(
        f'Research question: {question}\n\nGathered data:\n{findings}'
    )).output

## 7. Orchestrator — Plan → Execute → Validate → Replan? → Synthesize

In [ ]:
async def deep_search(question, tickers):
    additional_context = ''
    for attempt in range(MAX_REPLAN_ATTEMPTS + 1):
        print(f'\n[Phase 1] Planning (attempt {attempt + 1})...')
        user = question + (
            f'\n\nAdditional context:\n{additional_context}'
            if additional_context else ''
        )
        plan = (await planner_agent.run(user)).output
        print(format_plan(plan))

        print('\n[Phase 2] Executing...')
        results = await execute_plan(plan)

        print('\n[Phase 3] Validating...')
        v = validate_research(plan, results, tickers)
        print(format_validation(v))
        if v.is_valid:
            break

        if attempt < MAX_REPLAN_ATTEMPTS:
            issues = v.gaps + v.errors
            additional_context = (
                'Address these gaps/errors. Every required ticker must have a '
                'no-dependency financial_api task returning CompanyMetrics:\n'
                + '\n'.join(f'- {x}' for x in issues)
            )

    if not v.is_valid:
        raise RuntimeError(
            'Research validation failed; synthesis blocked: '
            + '; '.join(v.errors + v.gaps)
        )
    print('\n[Phase 4] Synthesizing...')
    return await synthesize_report(question, results, v)

## 8. Run and inspect

In [ ]:
question = (
    "Analyze NVIDIA's financial performance and competitive position in the "
    "AI chip market. Compare with AMD and Intel on revenue growth, profitability, "
    "and risk factors. Produce a structured investment memo."
)
tickers = ['NVDA', 'AMD', 'INTC']
report = await deep_search(question, tickers)

print(f'\nTitle: {report.title}')
print(f'Confidence: {report.confidence_score:.0%}')
print('\nExecutive Summary:\n' + report.executive_summary)
print('\nConclusion:\n' + report.conclusion)